# RSSM Weekly Model Validation and Interpretation

This notebook is for validating whether the RSSM/world model behaves as proposed, not for comparing predictive performance against baselines. The focus is interpretability and mechanism checks: KL regime surprise, cross-ticker attention allocation, latent regime geometry, counterfactual crowd-in/crowd-out, and residual dependence.

## Validation Scope

| Check | Research question | Expected signal |
|---|---|---|
| KL regime surprise | Does the posterior-prior gap spike during major attention regime shifts? | Elevated KL near COVID and GME weeks |
| Attention allocation | Does cross-ticker attention rise during systemic or narrative contagion events? | Higher off-diagonal coupling around shock weeks |
| Latent regimes | Does `z_t` organize weeks into meaningful market eras? | COVID / meme periods separate from stable periods |
| Counterfactual probing | Does increasing one ticker's latent attention affect related and unrelated tickers differently? | Meme names crowd in; unrelated large caps may crowd out |
| Residual dependence | Does the latent state absorb shared cross-ticker variation? | Bernoulli residual correlations are small after conditioning on `z_t` |

Predictive metric comparisons against ARIMA/VAR/LSTM are intentionally excluded.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
import inspect
import json
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import yaml
import duckdb

warnings.filterwarnings("ignore", category=UserWarning)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model").is_dir() and (candidate / "eval").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.vocab import Vocabulary
from data.dataset import FEATURE_COLS
from model.twit_wave import ModelConfig, TwitWave
from model.rssm import kl_divergence

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"torch = {torch.__version__}")

PROJECT_ROOT = C:\stocktwits_2026\StockTwit_WM
torch = 2.12.0.dev20260408+cu128


In [2]:
MODEL_DIR = PROJECT_ROOT / "external_models" / "twitwave-rssm-large"
DATA_DIR = PROJECT_ROOT / "data" / "processed_week"
NETWORK_DIR = DATA_DIR / "ticker_network_fact"
OUT_DIR = PROJECT_ROOT / "outputs" / "eval" / "model_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = MODEL_DIR / "config.yaml"
NORM_STATS_PATH = MODEL_DIR / "norm_stats.json"
CHECKPOINT_PATH = MODEL_DIR / "checkpoints" / "best.pt"
EMBEDDINGS_DIR = MODEL_DIR / "embeddings"

# Keep notebook reruns fast. Use "full" after the model loads cleanly.
RUN_MODE = "full"  # "smoke" or "full"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"MODEL_DIR = {MODEL_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"NETWORK_DIR = {NETWORK_DIR}")
print(f"OUT_DIR = {OUT_DIR}")
print(f"DEVICE = {DEVICE}")

MODEL_DIR = C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large
DATA_DIR = C:\stocktwits_2026\StockTwit_WM\data\processed_week
NETWORK_DIR = C:\stocktwits_2026\StockTwit_WM\data\processed_week\ticker_network_fact
OUT_DIR = C:\stocktwits_2026\StockTwit_WM\outputs\eval\model_validation
DEVICE = cuda


## Existing Evaluation API Audit

In [3]:
import eval.utils as eval_utils
import eval.predict as eval_predict
import eval.kl_analysis as eval_kl
import eval.attention_analysis as eval_attention
import eval.latent_clustering as eval_latent
import eval.counterfactual as eval_counterfactual
import eval.residual_correlation as eval_residual

objects_to_check = {
    "eval.utils.load_rssm": eval_utils.load_rssm,
    "eval.predict.Predictor": eval_predict.Predictor,
    "eval.kl_analysis.compute_spike_stats": eval_kl.compute_spike_stats,
    "eval.attention_analysis.diagonal_vs_offdiagonal": eval_attention.diagonal_vs_offdiagonal,
    "eval.latent_clustering.extract_latent_states": eval_latent.extract_latent_states,
    "eval.counterfactual.run_counterfactual": eval_counterfactual.run_counterfactual,
    "eval.residual_correlation.compute_residuals": eval_residual.compute_residuals,
}

for name, obj in objects_to_check.items():
    print(f"{name}: {inspect.signature(obj)}")

print("\nNote: this notebook uses local helper functions below so the validation logic is explicit and independent of baseline-comparison scripts.")

eval.utils.load_rssm: (model_dir: 'str | Path', vocab_size: 'int', device: 'torch.device') -> 'TwitWave'
eval.predict.Predictor: (model: 'TwitWave', vocab: 'Vocabulary', device: 'torch.device') -> 'None'
eval.kl_analysis.compute_spike_stats: (kl_series: 'np.ndarray', threshold_sigma: 'float' = 2.0) -> 'dict'
eval.attention_analysis.diagonal_vs_offdiagonal: (A_list: 'list[np.ndarray]') -> 'pd.DataFrame'
eval.latent_clustering.extract_latent_states: (model: 'TwitWave', features_seq: 'torch.Tensor', ticker_ids_seq: 'torch.Tensor', device: 'torch.device', window_k: 'int | None' = None) -> 'np.ndarray'
eval.counterfactual.run_counterfactual: (model: 'TwitWave', vocab: 'Vocabulary', features_seq: 'torch.Tensor', ticker_ids_seq: 'torch.Tensor', target_ticker: 'str', delta_log_attn: 'float', eval_tickers: 'list[str]', device: 'torch.device', window_k: 'int | None' = None) -> 'pd.DataFrame'
eval.residual_correlation.compute_residuals: (presence_logits: 'np.ndarray', presence_true: 'np.ndarray')

## Load Artifacts and Check Compatibility

In [4]:
required_paths = [
    CONFIG_PATH,
    NORM_STATS_PATH,
    CHECKPOINT_PATH,
    EMBEDDINGS_DIR / "e_dec.npy",
    EMBEDDINGS_DIR / "e_ret.npy",
    EMBEDDINGS_DIR / "ticker_to_idx.json",
    EMBEDDINGS_DIR / "idx_to_ticker.json",
]

missing = [p for p in required_paths if not p.exists()]
for p in required_paths:
    print(f"{'OK' if p.exists() else 'MISSING':7} {p.relative_to(PROJECT_ROOT)}")
if missing:
    raise FileNotFoundError("Run notebooks/4_a_RSSM_week_eval_prepare.ipynb first. Missing: " + ", ".join(map(str, missing)))

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
model_cfg = cfg["model"]
eval_cfg = cfg.get("eval", {})
norm_stats = {k: np.asarray(v, dtype=np.float32) for k, v in json.loads(NORM_STATS_PATH.read_text()).items()}
e_dec = np.load(EMBEDDINGS_DIR / "e_dec.npy")
e_ret = np.load(EMBEDDINGS_DIR / "e_ret.npy")

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = ckpt.get("model_state") or ckpt.get("model_state_dict") or ckpt.get("state_dict")
print("checkpoint keys:", list(ckpt.keys()))
print("checkpoint model_cfg:", ckpt.get("model_cfg"))
print("config model:", model_cfg)
print("embedding shapes:", e_dec.shape, e_ret.shape)
print("norm mean/std shapes:", norm_stats["mean"].shape, norm_stats["std"].shape)

OK      external_models\twitwave-rssm-large\config.yaml
OK      external_models\twitwave-rssm-large\norm_stats.json
OK      external_models\twitwave-rssm-large\checkpoints\best.pt
OK      external_models\twitwave-rssm-large\embeddings\e_dec.npy
OK      external_models\twitwave-rssm-large\embeddings\e_ret.npy
OK      external_models\twitwave-rssm-large\embeddings\ticker_to_idx.json
OK      external_models\twitwave-rssm-large\embeddings\idx_to_ticker.json
checkpoint keys: ['model_state', 'optimizer_state', 'lr_sched_state', 'beta_sched_state', 'global_step', 'epoch', 'best_val_loss', 'best_val_recon', 'epochs_no_improve', 'model_cfg', 'train_cfg', 'kl_log']
checkpoint model_cfg: {'vocab_size': 1521, 'embed_dim': 128, 'd_enc': 512, 'h_dim': 1024, 's_dim': 256, 'n_heads': 16, 'n_layers': 4, 'window_k': 8, 'mlp_hidden': 512, 'feature_dim': 5, 'top_k': 100, 'dropout': 0.15}
config model: {'d_enc': 512, 'dropout': 0.15, 'embed_dim': 128, 'feature_dim': 5, 'h_dim': 1024, 'mlp_hidden': 512, 'n_

In [5]:
def infer_gru_input_extra_dim(state_dict: dict, s_dim: int) -> int:
    weight = state_dict.get("rssm.gru.weight_ih")
    if weight is None:
        return 0
    return max(0, int(weight.shape[1]) - int(s_dim))

def make_model_config(model_cfg: dict, vocab_size: int, state_dict: dict | None = None) -> ModelConfig:
    extra_dim = infer_gru_input_extra_dim(state_dict or {}, int(model_cfg["s_dim"]))
    return ModelConfig(
        vocab_size=vocab_size,
        embed_dim=int(model_cfg.get("embed_dim", e_dec.shape[1])),
        d_enc=int(model_cfg["d_enc"]),
        h_dim=int(model_cfg["h_dim"]),
        s_dim=int(model_cfg["s_dim"]),
        n_heads=int(model_cfg["n_heads"]),
        n_layers=int(model_cfg["n_layers"]),
        window_k=int(model_cfg["window_k"]),
        mlp_hidden=int(model_cfg["mlp_hidden"]),
        feature_dim=int(model_cfg["feature_dim"]),
        top_k=int(model_cfg["top_k"]),
        dropout=0.0,
        gru_input_extra_dim=extra_dim,
    )

def normalize_state_dict_keys(sd: dict) -> dict:
    replacements = {
        "rssm.posterior_net.3.": "rssm.posterior_net.2.",
        "rssm.posterior_net.6.": "rssm.posterior_net.4.",
        "rssm.prior_net.3.": "rssm.prior_net.2.",
        "rssm.prior_net.6.": "rssm.prior_net.4.",
    }
    normalized = {}
    for key, value in sd.items():
        new_key = key.removeprefix("module.")
        for old, new in replacements.items():
            if new_key.startswith(old):
                new_key = new + new_key[len(old):]
                break
        normalized[new_key] = value
    return normalized

vocab_size_from_embeddings = int(e_dec.shape[0])
cleaned_state = normalize_state_dict_keys(state_dict)
tw_cfg = make_model_config(ckpt.get("model_cfg", model_cfg), vocab_size_from_embeddings, cleaned_state)
model = TwitWave(tw_cfg)
local_state = model.state_dict()

shape_mismatches = []
for key, value in cleaned_state.items():
    if key in local_state and tuple(value.shape) != tuple(local_state[key].shape):
        shape_mismatches.append((key, tuple(value.shape), tuple(local_state[key].shape)))

MODEL_READY = False
load_result = None
if shape_mismatches:
    print("Model is NOT ready for validation. Shape mismatches:")
    for key, ckpt_shape, local_shape in shape_mismatches:
        print(f"  {key}: checkpoint={ckpt_shape}, local={local_shape}")
else:
    load_result = model.load_state_dict(cleaned_state, strict=False)
    MODEL_READY = len(load_result.missing_keys) == 0 and len(load_result.unexpected_keys) == 0
    model.to(DEVICE).eval()
    print("missing keys:", load_result.missing_keys)
    print("unexpected keys:", load_result.unexpected_keys)

print("MODEL_READY =", MODEL_READY)
if not MODEL_READY:
    print("Interpretability cells below will skip model-dependent computation until the RSSM code/checkpoint mismatch is fixed.")

missing keys: []
unexpected keys: []
MODEL_READY = True


## Load Weekly Panels

In [6]:
vocab_path = DATA_DIR / "vocab.json"
vocab = Vocabulary.load(vocab_path)

panels = {}
for split in ["train", "val", "test1", "test2"]:
    path = DATA_DIR / f"panel_{split}.parquet"
    if path.exists():
        panel = pd.read_parquet(path)
        panel["week"] = pd.to_datetime(panel["week"])
        panels[split] = panel.sort_values(["week", "symbol"]).reset_index(drop=True)
        print(f"{split:5}: rows={len(panel):6}, weeks={panel['week'].nunique():3}, range={panel['week'].min().date()} to {panel['week'].max().date()}")
    else:
        print(f"{split:5}: missing {path}")

assert "train" in panels, "Training panel is needed for context warm-up."

[vocab] loaded 1520 tickers from C:\stocktwits_2026\StockTwit_WM\data\processed_week\vocab.json
train: rows=109737, weeks=554, range=2008-05-26 to 2018-12-31
val  : rows= 10400, weeks= 52, range=2019-01-07 to 2019-12-30
test1: rows=  5200, weeks= 26, range=2020-01-06 to 2020-06-29
test2: rows=  7800, weeks= 39, range=2020-10-05 to 2021-06-28


## Observed Ticker-Ticker Fact Network

In [7]:
NETWORK_SUMMARY_PATH = NETWORK_DIR / "weekly_network_summary.parquet"
NETWORK_EDGES_PATH = NETWORK_DIR / "weekly_ticker_edges.parquet"
NETWORK_NODES_PATH = NETWORK_DIR / "weekly_ticker_nodes.parquet"

for path in [NETWORK_SUMMARY_PATH, NETWORK_EDGES_PATH, NETWORK_NODES_PATH]:
    print(f"{'OK' if path.exists() else 'MISSING':7} {path.relative_to(PROJECT_ROOT)}")

if not NETWORK_EDGES_PATH.exists():
    print("Run scripts/2_c_week_ticker_network_fact.py before attention validation against observed user/post networks.")
else:
    con = duckdb.connect()
    summary = con.execute(f"""
        SELECT
            COUNT(*) AS n_weeks,
            MIN(week) AS min_week,
            MAX(week) AS max_week,
            SUM(n_nodes) AS node_week_rows,
            SUM(n_edges) AS edge_week_rows,
            SUM(message_co_mentions) AS message_co_mentions,
            SUM(user_week_co_mentions) AS user_week_co_mentions
        FROM read_parquet('{NETWORK_SUMMARY_PATH.as_posix()}')
    """).fetchdf()
    display(summary)

    event_edges = con.execute(f"""
        SELECT week, ticker_i, ticker_j, message_co_mentions, user_week_co_mentions, total_weight
        FROM read_parquet('{NETWORK_EDGES_PATH.as_posix()}')
        WHERE week IN (DATE '2020-02-17', DATE '2020-03-23', DATE '2021-01-25')
        ORDER BY week, total_weight DESC
        LIMIT 30
    """).fetchdf()
    display(event_edges)
    con.close()

OK      data\processed_week\ticker_network_fact\weekly_network_summary.parquet
OK      data\processed_week\ticker_network_fact\weekly_ticker_edges.parquet
OK      data\processed_week\ticker_network_fact\weekly_ticker_nodes.parquet


,n_weeks,min_week,max_week,node_week_rows,edge_week_rows,message_co_mentions,user_week_co_mentions
0,762,2008-05-26,2022-12-26,151337.0,11159469.0,46521911.0,140513569.0


,week,ticker_i,ticker_j,message_co_mentions,user_week_co_mentions,total_weight
0,2020-02-17,SPCE,TSLA,1935.0,1807.0,3742.0
1,2020-02-17,AAPL,SPY,1878.0,1087.0,2965.0
2,2020-02-17,QQQ,SPY,2124.0,682.0,2806.0
3,2020-02-17,SPY,TSLA,1354.0,1264.0,2618.0
4,2020-02-17,SPCE,SPY,1057.0,1266.0,2323.0
5,2020-02-17,AAPL,TSLA,898.0,994.0,1892.0
6,2020-02-17,AAPL,SPCE,487.0,854.0,1341.0
7,2020-02-17,AAPL,MSFT,714.0,486.0,1200.0
8,2020-02-17,AAPL,AMZN,673.0,506.0,1179.0
9,2020-02-17,SPX,SPY,882.0,249.0,1131.0


This fact network is the observed benchmark for the learned attention matrix. A good attention-allocation validation should compare high-attention model edges against high-weight observed co-mention/co-user edges in the same week, and compare event-week coupling against stable-week coupling.

## Local Validation Helpers

In [8]:
FEATURE_NAMES = list(FEATURE_COLS)

def build_week_tensors(panel: pd.DataFrame, weeks: list[pd.Timestamp], top_k: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    feat_list, ids_list, pres_list = [], [], []
    grouped = {pd.Timestamp(w): g for w, g in panel.groupby("week")}
    for week in weeks:
        grp = grouped.get(pd.Timestamp(week))
        feat = np.zeros((top_k, len(FEATURE_NAMES)), dtype=np.float32)
        ids = np.zeros(top_k, dtype=np.int64)
        pres = np.zeros(vocab.size, dtype=np.float32)
        if grp is not None and len(grp) > 0:
            grp = grp.sort_values("log_attention", ascending=False).head(top_k)
            values = grp[FEATURE_NAMES].to_numpy(np.float32)
            values = (values - norm_stats["mean"]) / norm_stats["std"]
            n = len(grp)
            feat[:n] = values
            ids[:n] = [vocab.encode(s) for s in grp["symbol"].tolist()]
            pres[ids[:n]] = 1.0
        feat_list.append(feat)
        ids_list.append(ids)
        pres_list.append(pres)
    return torch.tensor(np.stack(feat_list)), torch.tensor(np.stack(ids_list)), torch.tensor(np.stack(pres_list))

def trace_posterior(model: TwitWave, features: torch.Tensor, ticker_ids: torch.Tensor) -> pd.DataFrame:
    model.eval()
    features = features.to(DEVICE)
    ticker_ids = ticker_ids.to(DEVICE)
    h, s = model.rssm.init_state(1, DEVICE)
    a_cache = []
    rows = []
    with torch.no_grad():
        for t in range(features.shape[0]):
            feat_t = features[t].unsqueeze(0)
            ids_t = ticker_ids[t].unsqueeze(0)
            a_t, A_t = model._encode_step(feat_t, ids_t)
            a_cache.append(a_t)
            window = model._build_window(a_cache, model.cfg.window_k, DEVICE)
            e_t = model.temporal_enc(window)
            h = model.rssm.gru_step(h, s)
            s, post_mean, post_logvar = model.rssm.posterior(h, e_t)
            _, prior_mean, prior_logvar = model.rssm.prior(h)
            kl = kl_divergence(post_mean, post_logvar, prior_mean, prior_logvar)
            A = A_t[0].detach().cpu().numpy()
            n = A.shape[0]
            diag = float(np.trace(A) / max(n, 1))
            off = float((A.sum() - np.trace(A)) / max(n * n - n, 1))
            z = torch.cat([h, s], dim=-1)[0].detach().cpu().numpy()
            rows.append({"step": t, "kl": float(kl.item()), "attn_diag": diag, "attn_offdiag": off, "coupling_ratio": off / (diag + 1e-8), "z": z})
    return pd.DataFrame(rows)

def context_to_week(panel_all: pd.DataFrame, week: str | pd.Timestamp, context_len: int) -> tuple[list[pd.Timestamp], torch.Tensor, torch.Tensor]:
    week = pd.Timestamp(week)
    weeks = sorted(panel_all.loc[panel_all["week"] <= week, "week"].unique())[-context_len:]
    features, ids, _ = build_week_tensors(panel_all, weeks, top_k=tw_cfg.top_k)
    return weeks, features, ids

def run_latent_counterfactual(model: TwitWave, features: torch.Tensor, ids: torch.Tensor, target: str, eval_tickers: list[str], delta: float) -> pd.DataFrame:
    model.eval()
    features = features.to(DEVICE)
    ids = ids.to(DEVICE)
    h, s = model.context_phase(features, ids)
    z = torch.cat([h, s], dim=-1).detach().requires_grad_(True)
    eval_ids = torch.tensor([[vocab.encode(t) for t in eval_tickers]], dtype=torch.long, device=DEVICE)
    target_id = torch.tensor([[vocab.encode(target)]], dtype=torch.long, device=DEVICE)
    original = model.decode_features(z.detach(), eval_ids)[0].detach().cpu().numpy()
    target_log_attn = model.decode_features(z, target_id)[0, 0, 0]
    target_log_attn.backward()
    grad = z.grad.detach()
    z_cf = z.detach() + delta * grad / (grad.norm() + 1e-8)
    perturbed = model.decode_features(z_cf, eval_ids)[0].detach().cpu().numpy()
    rows = []
    for i, ticker in enumerate(eval_tickers):
        for j, feat in enumerate(FEATURE_NAMES):
            rows.append({"ticker": ticker, "feature": feat, "original": float(original[i, j]), "perturbed": float(perturbed[i, j]), "delta": float(perturbed[i, j] - original[i, j])})
    return pd.DataFrame(rows)

## Check 1: KL Regime Surprise

In [9]:
if not MODEL_READY:
    print("SKIP: model checkpoint is not compatible with current repo model code.")
else:
    validation_panels = pd.concat([panels[k] for k in ["train", "val", "test1", "test2"] if k in panels], ignore_index=True)
    context_len = int(eval_cfg.get("context_len", 52))
    event_weeks = ["2020-02-20", "2020-03-23", "2021-01-22", "2021-01-28"]
    rows = []
    for event_week in event_weeks:
        weeks, feat, ids = context_to_week(validation_panels, event_week, context_len)
        trace = trace_posterior(model, feat, ids)
        rows.append({"event_week": event_week, "context_start": str(pd.Timestamp(weeks[0]).date()), "kl_last": trace["kl"].iloc[-1], "kl_context_mean": trace["kl"].mean(), "kl_context_z": (trace["kl"].iloc[-1] - trace["kl"].mean()) / (trace["kl"].std() + 1e-8)})
    kl_event_df = pd.DataFrame(rows)
    display(kl_event_df)
    kl_event_df.to_csv(OUT_DIR / "kl_event_alignment_smoke.csv", index=False)

,event_week,context_start,kl_last,kl_context_mean,kl_context_z
0,2020-02-20,2019-02-25,53.260124,57.881208,-0.352266
1,2020-03-23,2019-04-01,53.303059,57.173838,-0.270852
2,2021-01-22,2019-10-28,53.694611,58.194360,-0.355988
3,2021-01-28,2019-11-04,54.454735,57.641866,-0.256272


Interpretation: KL is a surprise signal. The validation target is not low KL everywhere; it is selective elevation when the observed attention ecosystem forces the posterior away from the prior.

## Check 2: Cross-Ticker Attention Allocation

In [10]:
if not MODEL_READY:
    print("SKIP: model checkpoint is not compatible with current repo model code.")
else:
    validation_panels = pd.concat([panels[k] for k in ["train", "val", "test1", "test2"] if k in panels], ignore_index=True)
    rows = []
    for label, week in {"stable_val": "2019-06-21", "covid": "2020-02-20", "gme": "2021-01-22"}.items():
        weeks, feat, ids = context_to_week(validation_panels, week, int(eval_cfg.get("context_len", 52)))
        trace = trace_posterior(model, feat, ids)
        last = trace.iloc[-1]
        rows.append({"label": label, "week": week, "attn_diag": last["attn_diag"], "attn_offdiag": last["attn_offdiag"], "coupling_ratio": last["coupling_ratio"]})
    attention_df = pd.DataFrame(rows)
    display(attention_df)
    attention_df.to_csv(OUT_DIR / "attention_event_coupling_smoke.csv", index=False)

,label,week,attn_diag,attn_offdiag,coupling_ratio
0,stable_val,2019-06-21,0.01,0.01,1.000002
1,covid,2020-02-20,0.01,0.01,0.999999
2,gme,2021-01-22,0.01,0.01,1.000000


Interpretation: diagonal mass is a proxy for intrinsic ticker dynamics; off-diagonal mass is a proxy for ecosystem coupling. Event weeks should be inspected relative to stable weeks, not as standalone numbers.

## Check 3: Latent Regime Geometry

In [11]:
def era_label(week) -> str:
    w = pd.Timestamp(week)
    if w < pd.Timestamp("2013-01-01"):
        return "early"
    if w < pd.Timestamp("2017-01-01"):
        return "maturity"
    if w < pd.Timestamp("2020-01-01"):
        return "pre_covid"
    if w < pd.Timestamp("2021-01-01"):
        return "covid"
    if w < pd.Timestamp("2022-01-01"):
        return "meme"
    return "post_meme"

if not MODEL_READY:
    print("SKIP: model checkpoint is not compatible with current repo model code.")
else:
    from sklearn.metrics import silhouette_score
    validation_panels = pd.concat([panels[k] for k in ["train", "val", "test1", "test2"] if k in panels], ignore_index=True)
    all_weeks = sorted(validation_panels["week"].unique())
    if RUN_MODE == "smoke":
        sample_weeks = [w for w in all_weeks if pd.Timestamp(w).year in [2019, 2020, 2021]][::4]
    else:
        sample_weeks = all_weeks
    z_rows = []
    for week in sample_weeks:
        weeks, feat, ids = context_to_week(validation_panels, week, int(eval_cfg.get("context_len", 52)))
        trace = trace_posterior(model, feat, ids)
        z_rows.append({"week": pd.Timestamp(week), "era": era_label(week), "z": trace["z"].iloc[-1]})
    Z = np.stack([r["z"] for r in z_rows])
    eras = pd.Series([r["era"] for r in z_rows])
    era_codes = eras.astype("category").cat.codes.to_numpy()
    sil = silhouette_score(Z, era_codes) if len(set(era_codes)) > 1 and len(Z) > len(set(era_codes)) else np.nan
    latent_summary = pd.DataFrame({"era": eras}).value_counts().rename("n_weeks").reset_index()
    print("latent matrix shape:", Z.shape)
    print("era silhouette:", sil)
    display(latent_summary)
    pd.DataFrame({"week": [r["week"] for r in z_rows], "era": eras}).to_csv(OUT_DIR / "latent_era_sample.csv", index=False)

latent matrix shape: (671, 1280)
era silhouette: 0.15491026639938354


,era,n_weeks
0,early,241
1,maturity,208
2,pre_covid,157
3,covid,39
4,meme,26


Interpretation: this check asks whether the learned latent state has regime structure. A useful next full run should add UMAP/t-SNE plots after the checkpoint loads exactly.

## Check 4: Counterfactual Crowd-In / Crowd-Out

In [12]:
COUNTERFACTUALS = [
    {"name": "gme_squeeze", "target": "GME", "week": "2021-01-22", "delta": 3.0, "tickers": ["GME", "AMC", "BB", "NOK", "TSLA", "AAPL", "MSFT", "SPY", "AMZN", "NFLX"]},
    {"name": "covid_market_shock", "target": "SPY", "week": "2020-02-20", "delta": -3.0, "tickers": ["SPY", "QQQ", "XLF", "XLE", "GLD", "TLT", "VIX", "AAPL", "AMZN", "TSLA"]},
]

if not MODEL_READY:
    print("SKIP: model checkpoint is not compatible with current repo model code.")
else:
    validation_panels = pd.concat([panels[k] for k in ["train", "val", "test1", "test2"] if k in panels], ignore_index=True)
    for exp in COUNTERFACTUALS:
        tickers = [t for t in exp["tickers"] if vocab.has(t)]
        weeks, feat, ids = context_to_week(validation_panels, exp["week"], int(eval_cfg.get("context_len", 52)))
        cf_df = run_latent_counterfactual(model, feat, ids, exp["target"], tickers, exp["delta"])
        log_df = cf_df[cf_df["feature"] == "log_attention"].sort_values("delta", ascending=False)
        print("\n", exp["name"])
        display(log_df)
        cf_df.to_csv(OUT_DIR / f"counterfactual_{exp['name']}.csv", index=False)


 gme_squeeze


,ticker,feature,original,perturbed,delta
25,AAPL,log_attention,2.085352,2.433904,0.348552
10,BB,log_attention,1.513900,1.855325,0.341425
40,AMZN,log_attention,1.318737,1.658092,0.339355
20,TSLA,log_attention,0.960145,1.294204,0.334060
5,AMC,log_attention,0.959961,1.294018,0.334056
15,NOK,log_attention,0.955448,1.289220,0.333771
45,NFLX,log_attention,0.938721,1.272387,0.333665
30,MSFT,log_attention,0.912632,1.246026,0.333394
35,SPY,log_attention,0.840603,1.172751,0.332148
0,GME,log_attention,0.746112,1.076041,0.329929



 covid_market_shock


,ticker,feature,original,perturbed,delta
0,SPY,log_attention,0.850042,0.527279,-0.322764
25,TLT,log_attention,0.969567,0.644241,-0.325325
30,VIX,log_attention,0.969785,0.644456,-0.325329
45,TSLA,log_attention,0.969913,0.644580,-0.325332
15,XLE,log_attention,0.970186,0.644849,-0.325337
5,QQQ,log_attention,0.970109,0.644772,-0.325337
10,XLF,log_attention,0.970285,0.644944,-0.325341
40,AMZN,log_attention,1.329471,0.996780,-0.332691
20,GLD,log_attention,1.397664,1.063782,-0.333882
35,AAPL,log_attention,2.096913,1.750176,-0.346737


Interpretation: the sign pattern matters more than the absolute magnitude. For GME, related meme tickers should be inspected separately from broad-market names.

## Check 5: Residual Dependence Diagnostic

In [13]:
if not MODEL_READY:
    print("SKIP: model checkpoint is not compatible with current repo model code.")
else:
    validation_panels = pd.concat([panels[k] for k in ["train", "val", "test1", "test2"] if k in panels], ignore_index=True)
    split = "test2" if "test2" in panels else "val"
    weeks = sorted(panels[split]["week"].unique())
    if RUN_MODE == "smoke":
        weeks = weeks[: min(len(weeks), 8)]
    residual_rows = []
    with torch.no_grad():
        for week in weeks:
            ctx_weeks, feat, ids = context_to_week(validation_panels, week, int(eval_cfg.get("context_len", 52)))
            h, s = model.context_phase(feat.to(DEVICE), ids.to(DEVICE))
            h, s, z, logits = model.forward_step_prior(h, s, use_mean=True)
            true_week = panels[split][panels[split]["week"] == week]
            y = np.zeros(vocab.size, dtype=np.float32)
            y[[vocab.encode(sym) for sym in true_week["symbol"].tolist()]] = 1.0
            probs = torch.sigmoid(logits[0]).cpu().numpy()
            residual_rows.append(y - probs)
    residuals = np.stack(residual_rows)
    active = np.where(np.abs(residuals).sum(axis=0) > 0)[0][:50]
    corr = np.corrcoef(residuals[:, active].T)
    offdiag = np.abs(corr[~np.eye(corr.shape[0], dtype=bool)]).mean()
    print(f"split={split}, weeks={len(weeks)}, active_tickers={len(active)}, mean_abs_offdiag_residual_corr={offdiag:.4f}")
    labels = [vocab.decode(int(i)) for i in active]
    pd.DataFrame(corr, index=labels, columns=labels).to_csv(OUT_DIR / f"residual_corr_{split}_smoke.csv")

split=test2, weeks=39, active_tickers=50, mean_abs_offdiag_residual_corr=0.5094


Interpretation: high residual correlation means the latent state is missing shared ticker variation. Low residual correlation supports the claim that `z_t` is acting as a sufficient state for the joint presence distribution.

## Current Blocker and Next Step

The validation logic above is ready, but it depends on an exact checkpoint load. The current Hugging Face checkpoint has one parameter shape mismatch with the local model code: `rssm.gru.weight_ih` is wider in the checkpoint than in the current `RSSM` implementation. Before interpreting any KL, attention, latent, or counterfactual result, reconcile that architecture difference and move the artifact-aware loader into `eval/utils.py` so the scripts and notebooks share one loading path.